# 01. Clean and feature-build

**Input:** `../data/raw.pkl`.
**Does:** restricts to U.S. cases; codes the ordinal **severity** scale and the **actor** type from the
free-text `Outcome`; builds model features; performs a court-level **merge** (caseload per court) with
before/after diagnostics.
**Output:** `../data/clean.pkl`.

Coding functions are defined at the top.

In [ ]:
import pandas as pd
import numpy as np

# ---------- functions (defined at top) ----------
def severity(o):
    """Free-text disposition -> ordinal severity tier 0-4 (most severe tier present)."""
    if pd.isna(o): return 0
    s = str(o).lower()
    if any(k in s for k in ["bar referr","referral","suspen","disqualif"," dq","pro hac","dismiss",
        "contempt","vexatious","filing bar","barred","default judgment","censure","grievance",
        "disciplinary","revocation","revoked","terminating"]): return 4
    if any(k in s for k in ["monetary","fine","costs order","cost order","adverse cost","attorney fee",
        "fees and cost","$","damages","disgorge"]): return 3
    if any(k in s for k in ["show cause","order to explain","struck","stricken","strike","waived",
        "ignored","disregard","certif","cle","disclos","corrected","refile","amend","notify client",
        "order to","sworn","affidavit"]): return 2
    if any(k in s for k in ["warning","admonish","caution","reprimand","rebuke"]): return 1
    return 0

def actor(p):
    """Party(ies) -> responsible actor type."""
    if pd.isna(p): return None
    if "Judge" in p: return "Judge"
    if p == "Pro Se Litigant": return "Pro se"
    if p in {"Lawyer","Governement Lawyer","Prosecutor","Federal Defender"}: return "Counseled"
    return None

def is_federal(court):
    """Heuristic: does the court string look like a U.S. federal court?"""
    return bool(pd.notna(court) and pd.Series([court]).str.contains(
        r"\.D\.|Cir\.|Bankr|Fed\.|District of", regex=True).iloc[0])

In [ ]:
df = pd.read_pickle("../data/raw.pkl")
us = df[df["State(s)"] == "USA"].copy()
print("US rows:", len(us))

us["severity"]  = us["Outcome"].apply(severity)
us["actor"]     = us["Party(ies)"].apply(actor)
us["federal"]   = us["Court"].apply(is_federal).astype(int)
us["date"]      = pd.to_datetime(us["Date"], errors="coerce")
us["year"]      = us["date"].dt.year
us["tool_named"]= (~us["AI Tool"].fillna("").isin(["","Implied","implied","Unidentified"])).astype(int)
top = ["contract","civil rights","tort","employment","administrative","family"]
us["field"] = us["Legal Field Primary"].where(us["Legal Field Primary"].isin(top), "other")

print("severity distribution:", us["severity"].value_counts().sort_index().to_dict())
print("actor distribution:", us["actor"].value_counts(dropna=False).to_dict())

### Merge: court-level caseload
Build a court-level summary and merge it back on. Print row counts **before and after** and check for
unmatched keys.

In [ ]:
court_load = us.groupby("Court").size().rename("court_caseload").reset_index()
print("before merge:", us.shape, "| court summary rows:", court_load.shape[0])

n_before = len(us)
us = us.merge(court_load, on="Court", how="left", validate="many_to_one")

print("after merge:", us.shape)
assert len(us) == n_before, "row count changed during merge!"
print("unmatched court_caseload (should be 0):", int(us["court_caseload"].isna().sum()))

In [ ]:
keep = ["Case Name","Court","date","year","actor","severity","federal","tool_named",
        "field","court_caseload","Professional Sanction","Monetary Penalty","Outcome"]
clean = us[keep].reset_index(drop=True)
clean.to_pickle("../data/clean.pkl")
print("wrote ../data/clean.pkl", clean.shape)